# Validation des données MAPEPI pour le TdB DSE RDC  
Esteban Montandon  
27 Feb 2026

In [ ]:
import re
from openhexa.sdk import current_run
import pandas as pd
import numpy as np

In [ ]:
# nouvelles_donnees_path = "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2024/2024_Database_Week20.csv"
# nouvelles_donnees_path = "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2025/2025_Database_Week22.csv"
# nouvelles_donnees_path = "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2025/2025_Database_Week36.csv"

In [ ]:
# charger les données
df = pd.read_csv(nouvelles_donnees_path, encoding = "latin-1", delimiter=";")

# Le fichier est déjà séparé
if len(df.columns) == 1:
    df = pd.read_csv(nouvelles_donnees_path, encoding = "latin-1", delimiter=",")    
    
# drop NA rows that were causing second test to fail
df.dropna(axis='index', subset='ZS', inplace=True)

In [ ]:
# # expected list
maladie_list = ['DRACUNCULOSE', 'MONKEYPOX', 'DIARRHEE DHY M5', 'FIEVRE TYPHOIDE',
       'GRIPPE', 'IRA', 'PALUDISME SUSP', 'PALUDISME CONF', 'PNEUMONIE',
       'FIEVRE JAUNE', 'CHOLERA', 'DIARR SANGLANTE', 'MENINGITE',
       'ROUGEOLE', 'COVID-19', 'MAPI GRAVES', 'MAPI LEGERES', 'PESTE',
       'PFA', 'RAGE', 'TETANOS MATERNE', 'TNN', 'CHIKUNGUNYA',
       'COQUELUCHE', 'DECES MATERNELS', 'DIPHTERIE', 'FHA']
  

### Testes de validation

In [ ]:
file_name = nouvelles_donnees_path.split("/")[-1]
week = int(re.findall(r"\d+", file_name)[1])

# 1) Max week in file == week of file name
if (df.NUMSEM.max() != week):
    raise ValueError("La semaine de données la plus récente ne correspond pas à la semaine du nom du fichier. "
                     "Veuillez vérifier que le fichier contient toutes les données prévues.")

# 2) N unique weeks in file == number of the week in file name
if (len(df.NUMSEM.unique()) != week):
    raise ValueError("Le fichier ne comporte pas toutes les semaines de données. "
                     "Veuillez vérifier que le fichier contient toutes les données prévues.")

In [ ]:
# 3) Greater than 75% of provinces reporting in total
reported = df[["PROV", "NUMSEM"]].drop_duplicates()
mean_prov_per_week = np.mean(reported.groupby("NUMSEM").size())
if (mean_prov_per_week < 20):
    raise ValueError("Il y a une proportion inhabituellement élevée de données manquantes (plus de 25 % des provinces par semaine). "
               "Veuillez vérifier que le fichier contient toutes les données prévues.")

In [ ]:
# 4) list of missing diseases (must be empty, otherwise -> error)
maladie_not_found = df.loc[~df["MALADIE"].isin(maladie_list) & df["MALADIE"].notna(), "MALADIE"].unique()

# Check names in the MALADIE column
if (len(maladie_not_found) > 0):
    raise ValueError("Il existe des maladies dans la COLONNE MALADIE qui ne sont pas reconnues: "
                     f"{maladie_not_found}. Veuillez vérifier la liste correcte des maladies: {maladie_list}")

In [ ]:
# 5) All diseases present in the file

# drop Na first
if (len(df.MALADIE.dropna().unique()) != 27):
    raise ValueError("Il manque des maladies dans l'ensemble de données. "
                     "Veuillez vérifier que le fichier contient toutes les données prévues.")